# 6.3 混淆矩阵与训练曲线分析

本 notebook 介绍混淆矩阵的计算与可视化，以及如何通过训练曲线诊断模型训练状态。

**学习目标：**
- 理解混淆矩阵的含义（行=真实标签，列=预测标签）
- 掌握从混淆矩阵计算精确率、召回率、准确率
- 学会绘制混淆矩阵
- 通过训练曲线识别过拟合、欠拟合等问题

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch 版本: {torch.__version__}")

## 1. 混淆矩阵基本概念

混淆矩阵（Confusion Matrix）是分类模型评估的核心工具。

对于 N 类分类问题，混淆矩阵是一个 N x N 的矩阵：
- **行（row）**：表示**真实标签**
- **列（column）**：表示**预测标签**
- **对角线元素**：正确分类的样本数
- **非对角线元素**：错误分类的样本数

**二分类示例：**

```
                预测
              正(P)  负(N)
真实  正(P)   TP     FN
      负(N)   FP     TN
```

- TP（True Positive）：真正例，正确预测为正
- FN（False Negative）：假反例，正样本被错误预测为负
- FP（False Positive）：假正例，负样本被错误预测为正
- TN（True Negative）：真反例，正确预测为负

## 2. 计算混淆矩阵

In [ ]:
def compute_confusion_matrix(y_true: torch.Tensor, y_pred: torch.Tensor, 
                              num_classes: int) -> torch.Tensor:
    """计算混淆矩阵。
    
    Args:
        y_true: 真实标签, shape (N,)
        y_pred: 预测标签, shape (N,)
        num_classes: 类别数
    
    Returns:
        混淆矩阵, shape (num_classes, num_classes)
        行=真实标签, 列=预测标签
    """
    cm = torch.zeros(num_classes, num_classes, dtype=torch.long)
    for t, p in zip(y_true, y_pred):
        cm[t.long(), p.long()] += 1
    return cm

# 模拟一个 4 类分类问题
torch.manual_seed(42)
num_classes = 4
n_samples = 200

# 真实标签
y_true = torch.randint(0, num_classes, (n_samples,))

# 模拟预测（大部分正确，少部分错误）
y_pred = y_true.clone()
# 随机翻转 20% 的预测
flip_mask = torch.rand(n_samples) < 0.2
y_pred[flip_mask] = torch.randint(0, num_classes, (flip_mask.sum(),))

# 计算混淆矩阵
cm = compute_confusion_matrix(y_true, y_pred, num_classes)
print("混淆矩阵:")
print(cm)
print(f"\n对角线（正确分类）: {cm.diag().tolist()}")
print(f"总正确数: {cm.diag().sum().item()}/{n_samples}")
print(f"总体准确率: {cm.diag().sum().item() / n_samples:.2%}")

## 3. 从混淆矩阵计算指标

**关键指标：**
- **精确率 (Precision)**: 预测为正中实际为正的比例 = TP / (TP + FP)
- **召回率 (Recall)**: 实际为正中被正确预测的比例 = TP / (TP + FN)
- **准确率 (Accuracy)**: 总体正确比例 = (TP + TN) / Total

In [ ]:
def compute_metrics(cm: torch.Tensor) -> dict:
    """从混淆矩阵计算各类别的精确率、召回率和 F1。
    
    Args:
        cm: 混淆矩阵, shape (num_classes, num_classes)
    
    Returns:
        包含各指标的字典
    """
    num_classes = cm.shape[0]
    cm_float = cm.float()
    
    # 每个类别的 TP
    tp = cm_float.diag()
    
    # 精确率: TP / (该列的总和) = TP / (TP + FP)
    col_sum = cm_float.sum(dim=0)  # 每列的和 = 预测为该类的总数
    precision = tp / col_sum.clamp(min=1e-8)
    
    # 召回率: TP / (该行的总和) = TP / (TP + FN)
    row_sum = cm_float.sum(dim=1)  # 每行的和 = 实际为该类的总数
    recall = tp / row_sum.clamp(min=1e-8)
    
    # F1 分数: 精确率和召回率的调和平均
    f1 = 2 * precision * recall / (precision + recall).clamp(min=1e-8)
    
    # 总体准确率
    accuracy = tp.sum() / cm_float.sum()
    
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy
    }

metrics = compute_metrics(cm)

class_names = ["猫", "狗", "鸟", "鱼"]
print(f"{'类别':<6} {'精确率':>8} {'召回率':>8} {'F1':>8} {'样本数':>8}")
print("-" * 45)
for i in range(num_classes):
    print(f"{class_names[i]:<6} {metrics['precision'][i]:>8.4f} "
          f"{metrics['recall'][i]:>8.4f} {metrics['f1'][i]:>8.4f} "
          f"{cm[i].sum().item():>8}")
print("-" * 45)
print(f"{'总体':>6} {metrics['precision'].mean():>8.4f} "
      f"{metrics['recall'].mean():>8.4f} {metrics['f1'].mean():>8.4f} "
      f"{cm.sum().item():>8}")
print(f"\n准确率: {metrics['accuracy']:.4f}")

## 4. 绘制混淆矩阵

In [ ]:
def plot_confusion_matrix(cm: torch.Tensor, class_names: list, 
                          normalize: bool = False):
    """绘制混淆矩阵热力图。
    
    Args:
        cm: 混淆矩阵
        class_names: 类别名称列表
        normalize: 是否按行归一化（显示比例）
    """
    if normalize:
        cm_display = cm.float() / cm.sum(dim=1, keepdim=True).clamp(min=1)
        fmt = ".2f"
        title = "归一化混淆矩阵 (按行)"
    else:
        cm_display = cm.float()
        fmt = ".0f"
        title = "混淆矩阵"
    
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_display.numpy(), cmap="Blues")
    
    # 添加数值标注
    n = len(class_names)
    for i in range(n):
        for j in range(n):
            value = cm_display[i, j].item()
            text_color = "white" if value > cm_display.max().item() / 2 else "black"
            text = f"{value:{fmt}}"
            ax.text(j, i, text, ha="center", va="center", color=text_color, fontsize=12)
    
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_title(title)
    
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    return fig

# 绘制原始和归一化的混淆矩阵
fig1 = plot_confusion_matrix(cm, class_names, normalize=False)
plt.show()

fig2 = plot_confusion_matrix(cm, class_names, normalize=True)
plt.show()

print("归一化混淆矩阵的每一行之和为 1")
print("对角线值越高，该类别的召回率越高")

## 5. 文本形式打印混淆矩阵

不依赖 matplotlib 的文本方式展示，适合在终端或日志中使用。

In [ ]:
def print_confusion_matrix(cm: torch.Tensor, class_names: list):
    """以文本形式打印混淆矩阵。
    
    Args:
        cm: 混淆矩阵
        class_names: 类别名称列表
    """
    n = len(class_names)
    max_name_len = max(len(name) for name in class_names)
    cell_width = max(5, max_name_len + 1)
    
    # 表头
    header = " " * (max_name_len + 3) + "预测"
    print(header)
    print(" " * (max_name_len + 3) + "  ".join(f"{name:>{cell_width}}" for name in class_names))
    print(" " * (max_name_len + 1) + "-" * ((cell_width + 2) * n + 2))
    
    # 每行
    for i in range(n):
        prefix = "真实 " if i == n // 2 else "     "
        row_str = prefix + f"{class_names[i]:>{max_name_len}} |"
        for j in range(n):
            val = cm[i, j].item()
            if i == j:
                row_str += f"  [{val:>{cell_width - 2}}]"
            else:
                row_str += f"  {val:>{cell_width}}"
        row_str += f"  | {cm[i].sum().item()}"
        print(row_str)
    
    print(" " * (max_name_len + 1) + "-" * ((cell_width + 2) * n + 2))
    
    # 准确率
    total = cm.sum().item()
    correct = cm.diag().sum().item()
    print(f"\n准确率: {correct}/{total} = {correct/total:.2%}")

print_confusion_matrix(cm, class_names)

## 6. 训练曲线分析

通过训练曲线（train loss vs val loss）可以诊断模型的训练状态。

In [ ]:
# 模拟三种典型训练情况
epochs = np.arange(100)

# 情况1: 正常训练（两者都下降）
train_loss_normal = 2.0 * np.exp(-0.04 * epochs) + 0.1 + np.random.normal(0, 0.02, 100)
val_loss_normal = 2.0 * np.exp(-0.03 * epochs) + 0.2 + np.random.normal(0, 0.03, 100)

# 情况2: 过拟合（train 下降，val 先降后升）
train_loss_overfit = 2.0 * np.exp(-0.05 * epochs) + 0.05 + np.random.normal(0, 0.02, 100)
val_loss_overfit = (
    2.0 * np.exp(-0.03 * epochs) + 0.3 
    + 0.005 * np.maximum(epochs - 30, 0)  # 30 epoch 后开始上升
    + np.random.normal(0, 0.03, 100)
)

# 情况3: 欠拟合（两者都高，下降缓慢）
train_loss_underfit = 2.0 * np.exp(-0.005 * epochs) + 1.0 + np.random.normal(0, 0.03, 100)
val_loss_underfit = 2.0 * np.exp(-0.004 * epochs) + 1.2 + np.random.normal(0, 0.04, 100)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 正常训练
axes[0].plot(epochs, train_loss_normal, label="Train Loss", alpha=0.8)
axes[0].plot(epochs, val_loss_normal, label="Val Loss", alpha=0.8)
axes[0].set_title("Normal Training", fontsize=14)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].set_ylim(0, 2.5)
axes[0].grid(True, alpha=0.3)

# 过拟合
axes[1].plot(epochs, train_loss_overfit, label="Train Loss", alpha=0.8)
axes[1].plot(epochs, val_loss_overfit, label="Val Loss", alpha=0.8)
axes[1].axvline(x=30, color="red", linestyle="--", alpha=0.5, label="Overfitting starts")
axes[1].set_title("Overfitting", fontsize=14)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].set_ylim(0, 2.5)
axes[1].grid(True, alpha=0.3)

# 欠拟合
axes[2].plot(epochs, train_loss_underfit, label="Train Loss", alpha=0.8)
axes[2].plot(epochs, val_loss_underfit, label="Val Loss", alpha=0.8)
axes[2].set_title("Underfitting", fontsize=14)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Loss")
axes[2].legend()
axes[2].set_ylim(0, 3.5)
axes[2].grid(True, alpha=0.3)

plt.suptitle("Training Curve Patterns", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 训练曲线诊断指南
print("训练曲线诊断指南:")
print("=" * 60)
print()
print("1. 正常训练:")
print("   - 训练损失和验证损失都在下降")
print("   - 验证损失略高于训练损失（正常的泛化差距）")
print("   - 两条曲线趋于平稳")
print("   - 应对: 继续训练或适当增加模型复杂度")
print()
print("2. 过拟合 (Overfitting):")
print("   - 训练损失持续下降")
print("   - 验证损失先降后升（分叉点为过拟合开始）")
print("   - 训练集和验证集之间差距越来越大")
print("   - 应对:")
print("     * 增加数据量 / 数据增强")
print("     * 添加正则化（Dropout, L2, Weight Decay）")
print("     * 减小模型容量")
print("     * 使用 Early Stopping")
print()
print("3. 欠拟合 (Underfitting):")
print("   - 训练损失和验证损失都很高")
print("   - 两者下降缓慢或几乎不下降")
print("   - 应对:")
print("     * 增加模型容量（更多层、更多参数）")
print("     * 增大学习率")
print("     * 增加训练轮数")
print("     * 检查数据预处理是否正确")

## 7. 实战：完整分类评估流程

In [ ]:
# 模拟一个完整的分类任务评估
torch.manual_seed(123)

# 模拟 5 类分类
class_names_5 = ["飞机", "汽车", "船", "火车", "自行车"]
n_classes = 5
n_test = 500

# 真实标签
y_true = torch.randint(0, n_classes, (n_test,))

# 模拟不同类别的预测准确率
# 某些类别容易混淆（比如汽车和火车）
y_pred = y_true.clone()
for i in range(n_test):
    true_label = y_true[i].item()
    rand_val = torch.rand(1).item()
    if true_label == 1 and rand_val < 0.15:  # 汽车容易被误判为火车
        y_pred[i] = 3
    elif true_label == 3 and rand_val < 0.15:  # 火车容易被误判为汽车
        y_pred[i] = 1
    elif rand_val < 0.08:  # 其他类别有少量误差
        y_pred[i] = torch.randint(0, n_classes, (1,))

# 计算混淆矩阵
cm_5 = compute_confusion_matrix(y_true, y_pred, n_classes)

# 打印文本混淆矩阵
print_confusion_matrix(cm_5, class_names_5)

# 绘制混淆矩阵
fig = plot_confusion_matrix(cm_5, class_names_5, normalize=True)
plt.show()

# 计算并打印指标
metrics_5 = compute_metrics(cm_5)
print(f"\n{'类别':<8} {'精确率':>8} {'召回率':>8} {'F1':>8}")
print("-" * 40)
for i in range(n_classes):
    print(f"{class_names_5[i]:<8} {metrics_5['precision'][i]:>8.4f} "
          f"{metrics_5['recall'][i]:>8.4f} {metrics_5['f1'][i]:>8.4f}")
print(f"\n总体准确率: {metrics_5['accuracy']:.4f}")

print("\n分析: 汽车和火车之间存在较高的混淆率，可能需要:")
print("- 增加这两类的训练数据")
print("- 使用更强的特征提取器")
print("- 进行数据增强以增加区分度")

## 8. 总结

### 混淆矩阵
- 行表示真实标签，列表示预测标签
- 对角线越大，分类效果越好
- 非对角线元素反映类别间的混淆程度

### 评估指标

| 指标 | 公式 | 含义 |
|------|------|------|
| 精确率 | TP / (TP + FP) | 预测为正的样本中真正为正的比例 |
| 召回率 | TP / (TP + FN) | 实际为正的样本中被正确识别的比例 |
| F1 | 2 * P * R / (P + R) | 精确率和召回率的调和平均 |
| 准确率 | 正确数 / 总数 | 总体分类正确的比例 |

### 训练曲线诊断

| 模式 | 训练损失 | 验证损失 | 解决方案 |
|------|----------|----------|----------|
| 正常 | 下降 | 下降 | 继续训练 |
| 过拟合 | 下降 | 先降后升 | 正则化、数据增强、Early Stop |
| 欠拟合 | 高且不降 | 高且不降 | 增加模型容量、增大学习率 |

---

## 练习题

**练习1（基础）：** 给定以下混淆矩阵，手动计算每个类别的精确率和召回率：
```
[[40,  5,  3],
 [ 2, 35,  8],
 [ 1,  6, 42]]
```

**练习2（进阶）：** 实现一个函数，输入训练和验证损失列表，自动判断属于哪种训练模式（正常/过拟合/欠拟合），并给出建议。

**练习3（挑战）：** 训练一个简单的分类器（如逻辑回归或小型 MLP），在训练结束后生成完整的评估报告，包括：混淆矩阵、各类别指标、训练曲线图。